# 从零开始动手实现 GPT 风格的语言模型

## 项目概要：通往智能文本的探索

想象一下，我们面前有一堆文字，一片混沌。如果我们要教会一台机器去理解它，去生成它，甚至去“思考”它，我们的第一步应该是什么？

**核心问题一：计算机如何“理解”文字？**

在数字世界里，文字仅仅是一串字符。如何将这些抽象的符号转化为机器能够处理的、富有意义的数字表示？这引出了我们的起点：**数据表示 (Data Representation)**，特别是如何构建一个“词典”，让每一个字符或词块都有其独特的身份。

**核心问题二：如何让机器“预测”下一个词？**

一旦文字有了数字身份，我们如何构建一个最简单的模型，让它根据已有的信息去猜测下一个可能出现的词？我们将从一个看似朴素但蕴含基本原理的**基线模型 (Baseline Model)** 开始，揭示机器学习语言的初步逻辑。

**核心问题三：如何超越“目光短浅”的预测？**

然而，仅仅根据最近的信息来预测下一个词是远远不够的。语言的精妙在于上下文。我们需要一种机制，让机器在做预测时，能够回顾并“关注”到所有相关的前文信息。这正是 **自注意力机制 (Self-Attention)** 的核心思想，它允许机器在处理每个词时，灵活地加权考虑历史上的每一个词的重要性。

**核心问题四：如何构建一个能“思考”的语言大脑？**

自注意力机制如同神经元之间的“连接”，但单个连接还不足以构成智慧。我们需要将这些连接组织起来，形成一个强大的“思考”单元。这涉及构建 **Transformer Block**，它结合了自注意力、前馈网络、残差连接和层归一化，将信息进行多维度加工。最终，我们将堆叠这些“思考单元”，构建出完整的 **GPT 风格语言模型**，赋予它更深层次的语言理解和生成能力。

**核心问题五：如何让机器适应不同任务？**

一个真正智能的语言模型不应仅仅局限于生成文本，它还应该能够解决各种实际问题，例如理解文本的情感、进行对话等等。我们也将探讨如何通过微调等方式，将预训练模型的强大能力**迁移到下游任务**中，让它变得更加实用和多才多艺。

通过这一系列的探索，我们将逐步揭开 Transformer 模型的神秘面纱，从最基本的砖块开始，亲手搭建起一个能够理解并生成复杂语言的智能系统。

---

## Phase 1: 数据表示 (Data Representation)

在构建任何复杂的架构（如 Transformer）之前，我们必须解决一个基本问题：**如何让计算机识别文本？**

想象一下，如果你只有一段文本字符串，计算机看到的只是二进制序列。为了让模型能够处理，我们需要构建一个字典（Vocabulary），将每个唯一的字符映射到一个整数 ID。这就是 **Tokenization** 的最简形式。

In [ ]:
from pathlib import Path
import urllib.request

file_path = Path("input.txt")
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"

if not file_path.exists():
    print("文件不存在，开始下载...")
    urllib.request.urlretrieve(url, file_path)
    print("下载完成！")
else:
    print("文件已存在，跳过下载。")

with open(file_path, 'r', encoding='utf-8') as f:
    text = f.read()

print(f"数据集总字符数: {len(text)}")

# 获取所有唯一字符并建立词表
chars = sorted(list(set(text)))  # 为什么sorted(list(set(text)))在这里能实现获取所有字符的功能。
vocab_size = len(chars)
print(f"词表大小: {vocab_size}")
print(f"所有字符: {''.join(chars)}")

Q: 为什么sorted(list(set(text)))在这里能实现获取所有字符的功能？

A: set（集合）是一个无序且元素唯一的数据结构。当你把整个文本扔进 set 时，它会自动丢弃所有重复的字符，只保留出现的每一个唯一字符。这就是我们获取“词表”的过程。如此得到的词表是英文字母 + 标点符号。

另外在 Python 中，sorted() 对字符串（字符）的默认排序方式是基于 Unicode 码位（Code Point） 的数值大小进行升序排列。在深度学习中，一致性（Consistency） 胜过一切。只要我们使用了 sorted()，无论你在哪台机器上运行，'a' 永远会被映射到同一个数字 ID。

### 练习 1.1

现在我们有了字符列表 `chars`。为了实现 `encode`（文本转数字）和 `decode`（数字转文本）的功能，我们需要建立两个映射表（Mapping tables）。

**TODO**: 请补全下面的代码，构建映射字典，并思考：为什么我们在这里选择**字符级**（Character-level）的 Tokenization 而不是单词级（Word-level）的？这对于词表大小（Vocab size）有什么影响？

In [ ]:
# TODO: 构建 string -> integer (stoi) 和 integer -> string (itos) 的映射

def encode(s):
    # TODO: 将字符串 s 转换为数字列表
    l = []
    for char in s:
        l.append(chars.index(char))
    return l

def decode(l):
    # TODO: 将数字列表 l 转换回字符串
    s = []
    for i in l:
        c = chars[i]
        s.append(c)
    s = ''.join(s)
    return s

# 参考答案
# def encode(s):
#     stoi = {c: i for i, c in enumerate(chars)}
#     return [stoi[c] for c in s]

# def decode(l):
#     itos = {i: c for i, c in enumerate(chars)}
#     return ''.join([itos[i] for i in l])

# 测试代码（请在补全后取消注释并运行）
test_str = "hello world"
encoded = encode(test_str)
print(f"Encoded: {encoded}")
print(f"Decoded: {decode(encoded)}")

### 练习 1.2: 数据张量化 (Tensors)

在 PyTorch 中，所有数据最终都必须以张量（Tensor）的形式存在。现在我们将整个莎士比亚数据集转换成一个巨大的整数序列。

In [ ]:
import torch

# TODO: 将整个数据集 encode 并转换为 torch.Tensor
# 别忘了用 torch.tensor() 包装 encode 后的结果
data = torch.tensor(encode(text), dtype=torch.long)

# 打印前 100 个字符的 Tensor 形式
print(data[:100])
print(f"Tensor 形状: {data.shape}")

**引导思考**：
将数据转换为 Tensor 后，我们就不再需要原始字符串了。在这个 Tensor 中，每个数字代表什么物理意义？为什么我们选择 `torch.long` 而不是 `torch.float`？

---

## Phase 2: 基准模型 (Baseline Model)

### 练习 2.1: 数据切分 (Train/Val Split)

为了评估模型质量，我们需要将数据集切分为**训练集**和**验证集**。

In [ ]:
# TODO: 计算切分点索引 (90% 训练, 10% 验证)
n = int(0.9 * len(data))
train_data = data[0:n-1]
val_data = data[n:]

print(f"训练集长度: {len(train_data)}")
print(f"验证集长度: {len(val_data)}")

### 练习 2.2: 构造批处理数据 (Data Batching)

我们需要从数据集中随机采样。对于输入序列 `x`，它的目标 `y` 就是该序列向后移动一个位置的序列。

In [ ]:
torch.manual_seed(1337)
batch_size = 4 # 每次并行处理多少个序列
block_size = 8 # 每个序列的最大长度

def get_batch(split, batch_size):
    # split 可以是 'train' 或 'val'
    data_subset = train_data if split == 'train' else val_data

    # TODO: 随机生成 batch_size 个起始索引
    ix = torch.randint(0, len(data_subset) - block_size, (batch_size,))

    # TODO: 根据 ix 提取 x 和 y
    x = [data_subset[i:i+block_size] for i in ix]
    y = [data_subset[i+1:i+block_size+1] for i in ix]
    return torch.stack(x), torch.stack(y)

xb, yb = get_batch('train', batch_size)
print('inputs (xb):')
print(xb.shape, xb)
print('targets (yb):')
print(yb.shape, yb)

**引导思考**：
观察 `xb` 的第一行和 `yb` 的第一行。如果 `xb[0]` 是 `[18, 47, 56]`，对应的 `yb[0]` 是什么？这种“错位偏移”的设计是如何体现模型“预测下一个字符”的任务目标的？

A: yb[0]=[47, 56, xx（下一个字符）]。当我们输入 x = 18 时，希望模型输出下一个字符，所以 y = 47。

**引导思考**：
当我们后续训练 Transformer 时，模型是一次性看完这 100 万个字符，还是“少食多餐”？如果切分点不是正好在句子的结尾，会有影响吗？

A: 模型要少食多餐。如果切分点不是正好在句子的结尾也没有问题，只要看的数据够多，就能掌握更大范围内的上下文关系。

### 练习 2.3: 二元语法模型 (Bigram Language Model)

我们将使用 PyTorch 构建一个最基础的模型。在这个模型中，Token 之间不会相互“交流”，每个 Token 只根据自己的索引去查找一个 Embedding 向量，这个向量直接代表了它对下一个字符出现的概率预测。（注意，token index 本身没有距离关系；embedding 向量才可能有距离关系。）

模型给出了对下一个字符的预测，但我们还需要判断：**它预测得准不准？**
例如，真实的下一个字符是 `e`。如果模型认为 `e` 的概率是 `0.9`，说明预测很好；如果模型认为 `e` 的概率只有 `0.1`，说明模型虽然给出了预测，但没有把高概率分配给正确答案。

因此，我们需要一个数值来衡量模型预测和真实答案之间的差距，这个数值就叫 **loss（损失）**。
loss 越大，说明模型预测越差；loss 越小，说明模型预测越接近真实答案。训练模型的过程，本质上就是不断调整参数，让 loss 逐渐变小。
在语言模型中，最常用的 loss 函数是 **交叉熵损失（Cross Entropy Loss）**。它的核心思想是：

**模型给正确答案分配的概率越高，loss 越小；模型给正确答案分配的概率越低，loss 越大。**

在 PyTorch 中，我们通常使用：

```python
F.cross_entropy(logits, targets)
```

其中，`logits` 是模型输出的原始分数，`targets` 是真实的下一个 token 的索引。需要注意的是，`F.cross_entropy` 内部会自动完成 softmax 相关计算，所以我们不需要先手动把 logits 转成概率。
对于 Bigram Language Model 来说，模型会根据当前 token 查表，得到对下一个 token 的预测分数。然后，我们用交叉熵损失比较这个预测和真实下一个 token 的差距，再通过反向传播更新这张表，使模型逐渐学会哪些字符更可能跟在哪些字符后面。


In [ ]:
import torch.nn as nn
from torch.nn import functional as F
from torch.distributions import Categorical

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # 每个 token 直接读取下一个 token 的 logits
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx 和 targets 都是 (B, T) 的张量
        # idx 是输入字符串的索引
        logits = self.token_embedding_table(idx) # (B, T, C)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx 是 (B, T) 的当前索引数组
        for _ in range(max_new_tokens):
            # TODO: 1. 获取预测结果 (logits)
            # TODO: 2. 只关注最后一个时间步的结果 (Focus on the last time step)
            # TODO: 3. 使用 Softmax 转化为概率
            # TODO: 4. 从概率分布中采样得到下一个 Token
            # TODO: 5. 将采样结果拼接到 idx 并继续循环

            # logits = self.token_embedding_table(idx)[:, -1, :]
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            m = Categorical(probs)
            idx_next = m.sample()
            idx = torch.cat((idx, idx_next.unsqueeze(1)), dim=1)

        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(f"Logits shape: {logits.shape}")
print(f"初始 Loss: {loss}")

In [ ]:
import torch

# 创建一个 (1, 1) 的张量，内容为 0（通常是换行符），作为生成的起点
idx = torch.zeros((1, 1), dtype=torch.long)

# 调用 generate 函数生成 100 个 token
# 注意：m.generate 返回的是数字索引，我们需要用 decode 转换回文字
generated_indices = m.generate(idx, max_new_tokens=100)[0].tolist()

print("--- 未经训练的模型生成的文本 ---")
print(decode(generated_indices))
print("------------------------------")

Q: nn.Embedding只是嵌入了，还没有到预测的阶段吧？

A: 通常情况下，nn.Embedding 只是把数字变成向量。但在我们的 Bigram 模型中，我们玩了一个数学“小花招”：

我们将 Embedding 的维度直接设为 vocab_size。这意味着，当你输入字符 'A' (ID: 13) 时，查出来的向量就是一个长度为 65 的数组。这个数组里的每一个数字，我们直接把它当成：如果当前是 'A'，下一个字符是词表中对应位置字符的“可能性”（Logits）。

Q: Embedding层为什么要做成(num_embeddings, embedding_dim)的矩阵形式？

A: 这种矩阵结构实际上是一个**可学习的查找表**。每一行代表一个 Token 的特征向量。通过矩阵形式，我们可以利用 GPU 的高效矩阵运算，同时让模型通过梯度下降来优化这些权重。在 Bigram 模型中，这行向量直接就是对下一个字符的预测分数。

Q: 为什么不用nn.Linear作embedding层？

A: 数学上它们是等价的（Embedding 等于 One-hot 输入乘以 Linear 层）。但在实现上，Embedding 通过“索引查找”代替了“矩阵乘法”，避免了处理高维稀疏矩阵，极大地节省了内存并提升了运算速度。

**引导思考**：
为什么在计算 Loss 时，我们需要对 `logits` 和 `targets` 进行 `view`（变形）处理？PyTorch 的 `cross_entropy` 对输入的维度有什么特殊要求？

A: 因为 `F.cross_entropy` 期望输入的 Logits 是二维的 `(Total_Samples, Classes)`，而我们的模型输出是三维的 `(Batch, Time, Classes)`。通过 `view(B*T, C)`，我们将每一个时间步的预测都看作一个独立的样本，从而符合 API 的规范。

### 练习 2.4: 训练模型 (Training the Model)

乱码是正常的！因为我们的 `token_embedding_table` 里的参数目前全是随机数。为了让它学会莎士比亚的风格，我们需要：
1. 创建一个优化器 (Optimizer)。
2. 循环获取 Batch 数据。
3. 计算 Loss 并反向传播更新权重。

**TODO**: 请使用 `torch.optim.AdamW` 优化器，并尝试写出一个简单的训练循环（例如 1000 个步长）。观察 Loss 是否会下降？

In [ ]:
import torch

# 创建优化器
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3, betas=(0.9, 0.9))

batch_size = 32
for steps in range(10000): # 10000 次训练迭代

    # 1. 获取随机 Batch 数据
    xb, yb = get_batch('train', batch_size)

    # 2. 前向传播计算 Loss
    logits, loss = m(xb, yb)

    # 3. 反向传播更新参数
    optimizer.zero_grad(set_to_none=True) # 清空梯度，推荐使用 set_to_none
    loss.backward()
    optimizer.step()

    if steps % 1000 == 0:
        print(f"Step {steps}: Loss {loss.item():.4f}")

print(f"最终 Loss: {loss.item():.4f}")

In [ ]:
# 训练后再次生成，看看效果
print("--- 训练后的模型生成的文本 ---")
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(context, max_new_tokens=200)[0].tolist()))

Q: 为什么 Loss 只能降到2.5左右?

A: 数学上的限制：Bigram 模型本质上是在学习一个条件概率表 $P(next|current)$。在莎士比亚数据集中，给定当前字符是 'e'，下一个字符可能是 ' ', 'a', 'n', 'r', 's' 等等。由于模型只看当前这一个字符，它无法区分这个 'e' 是属于 'the' 还是 'where' 或是 'hell'。这种固有的不确定性导致了它的 Loss 存在一个下界（理论上的最小熵）。

信息量的缺失：想象一下，如果我只给你看一个字母 'h'，让你猜下一个，你很难猜准；但如果我给你看 't', 'h'，你就更有把握猜出下一个是 'e'。Bigram 模型的“视野”太窄了（Context length = 1），所以它永远无法消除由于缺乏上下文而产生的那部分 Loss。

对比：目前 Loss 在 2.5 左右，这正是单字符预测的极限。当我们进化到 Self-Attention 机制时，模型能同时看到过去 8 个甚至上千个字符，那时 Loss 就能轻松降到 2.0 以下，生成的文本也会开始变得通顺。

---

## Phase 3: 自注意力机制 (Self-Attention)

### 3.1 核心矛盾：孤独的字符

在 Phase 2 的 Bigram 模型中，每个字符只看自己。例如预测 `the`，当模型看到 `h` 时，它不知道前面是 `t` 还是 `s`。为了生成通顺的文本，我们需要一种机制，让当前字符能**吸收**过去字符的信息。

### 3.2 信息融合的初级方案：历史平均 (Weighted Average)

最简单的办法是：让位置 $t$ 的输出等于从 $0$ 到 $t$ 所有输入向量的**平均值**。这样，$t$ 就“看到”了过去所有的信息。

但在 PyTorch 中，用 `for` 循环来做这件事非常慢。我们要用一个**数学小花招**：下三角矩阵乘法。

### 3.3 为什么不用 For 循环？ (性能验证)

虽然下面的矩阵乘法看起来有点绕，但它能极大地利用 GPU 的并行能力。我们对比一下方案 1（显式循环）和方案 2（矩阵乘法）的效率差异：

In [42]:
import torch
import time

# --- 核心对比：向量化 (Vectorization) 的威力 ---
B, T, C = 4, 8, 2

# 为了对比，我们需要一些随机输入数据 x
torch.manual_seed(42)
x = torch.randn(B, T, C)

# 1. 方案 1: 原始 For 循环
start = time.time()
for _ in range(100):
    xbow = torch.zeros((B, T, C))
    for b in range(B):
        for t in range(T):
            xprev = x[b, :t+1]
            xbow[b, t] = torch.mean(xprev, 0)
t1 = (time.time() - start) / 100

# 2. 方案 2: 矩阵乘法 (使用预计算的下三角权重)
# 这种方式利用了 PyTorch 的底层 C++/CUDA 优化
weights = torch.tril(torch.ones(T, T))
weights = weights / weights.sum(1, keepdim=True)

start = time.time()
for _ in range(100):
    xbow2 = weights @ x
t2 = (time.time() - start) / 100

# --- 输出打印 ---
print(f"[性能对比]")
print(f"方案 1 (For 循环): {t1:.6f}s")
print(f"方案 2 (矩阵乘法): {t2:.6f}s (快 {t1/t2:.1f} 倍)")

print(f"\n[一致性检查]")
# 再次强调：使用 atol=1e-7 来容忍微小的浮点数舍入误差
print(f"方案 1 与 2 结果是否一致: {torch.allclose(xbow, xbow2, atol=1e-7)}")

[性能对比]
方案 1 (For 循环): 0.000766s
方案 2 (矩阵乘法): 0.000012s (快 62.9 倍)

[一致性检查]
方案 1 与 2 结果是否一致: True


你会发现，矩阵乘法的速度通常比 `for` 循环快一个数量级以上。这就是为什么我们在实现神经网络（尤其是 Transformer）时，会竭尽全力把所有逻辑都转化为矩阵运算。

@ 符号在 PyTorch（以及 NumPy）中代表 矩阵乘法 (Matrix Multiplication)。它等同于调用 numpy.matmul() 函数。

在代码 weights @ x 中，它将形状为 (T, T) 的权重矩阵与形状为 (B, T, C) 的数据进行批量矩阵乘法。PyTorch 会自动处理 Batch 维度，最终得到 (B, T, C)。这实际上就是通过加权求和，让每个时间步都“看”到了它之前的历史信息。

### 3.4 信息融合：从数学公式到代码实现

在自注意力机制中，我们希望第 $t$ 个位置的字符能“融合”掉从 $1$ 到 $t$ 所有的信息。最简单的融合方式就是计算**过去所有时刻的平均值**。

**1. 数学表达**：
如果输入序列是 $x_1, x_2, \dots, x_T$，输出 $a_t$ 的计算如下：
$$a_t = \frac{1}{t} \sum_{i=1}^{t} x_i$$
这意味着每个时刻都在“吸收”它的历史，且权重是均匀分布的。

**2. 矩阵实现**：
在 PyTorch 中，我们通过一个**下三角归一化矩阵** $W$ 来实现这一过程：
$$A = W \cdot X$$

让我们以 $T=3$ 为例，看看 $W \cdot X$ 发生了什么：

$$W = \begin{pmatrix} 1 & 0 & 0 \\ 0.5 & 0.5 & 0 \\ 0.33 & 0.33 & 0.33 \end{pmatrix}, \quad X = \begin{pmatrix} x_1 \\ x_2 \\ x_3 \end{pmatrix}$$

当我们计算 $A = W \cdot X$ 时：
- **第一行**：$a_1 = (1 \cdot x_1 + 0 \cdot x_2 + 0 \cdot x_3) = x_1$
- **第二行**：$a_2 = (0.5 \cdot x_1 + 0.5 \cdot x_2 + 0 \cdot x_3) = \frac{x_1 + x_2}{2}$
- **第三行**：$a_3 = (0.33 \cdot x_1 + 0.33 \cdot x_2 + 0.33 \cdot x_3) = \frac{x_1 + x_2 + x_3}{3}$

你看！矩阵 $W$ 的每一行其实就是一个**掩码（Mask）**加**权重（Weight）**：
1.  **下三角结构**：保证了第 $t$ 行只能看到索引 $\le t$ 的数据（即“不看未来”）。
2.  **行归一化**：保证了所有看到的历史权重之和为 1（即“求平均”）。

In [43]:
import torch

# 假设 Batch=1, Time=3, Channels=2
B, T, C = 1, 3, 2
x = torch.tensor([[[10, 20], [30, 40], [50, 60]]], dtype=torch.float)

# 1. 创建下三角全1矩阵
tril = torch.tril(torch.ones(T, T))

# 2. 归一化：让每一行的和为 1，这样矩阵乘法就变成了“求平均”
weights = tril / tril.sum(1, keepdim=True)

# 3. 矩阵乘法融合信息
# xbow 的第 t 行 = x 中前 t 行的平均值
xbow = weights @ x

print("输入 x (3个时刻):\n", x)
print("\n权重矩阵 (下三角平均):\n", weights)
print("\n融合结果 xbow (各时刻融合了历史信息):\n", xbow)

输入 x (3个时刻):
 tensor([[[10., 20.],
         [30., 40.],
         [50., 60.]]])

权重矩阵 (下三角平均):
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

融合结果 xbow (各时刻融合了历史信息):
 tensor([[[10.0000, 20.0000],
         [20.0000, 30.0000],
         [30.0000, 40.0000]]])


### 3.5 终极方案：使用 Softmax 统一公式与代码

在深度学习中，我们很少直接做除法归一化，而是使用 `Softmax`。这种方法不仅数学上更优美，而且能处理未来我们将会遇到的“非均匀权重”问题。

**Softmax 方案的三个步骤：**
1.  **准备**：创建一个全 0 矩阵（表示所有历史同样重要）。
2.  **遮蔽 (Masking)**：将“未来”的位置（`tril == 0`）设为 $-\infty$。
3.  **激活 (Softmax)**：对行执行 `Softmax`。此时，$e^0=1$ 而 $e^{-\infty}=0$，每一行就自动变成了 $[1/t, 1/t, \dots, 0]$ 的形式。

这与你的公式 $a_t = \frac{1}{t} \sum_{i=1}^{t} x_i$ 是**完全等价**的。

In [44]:
import torch.nn.functional as F

# 1. 初始化全 0 权重
tril = torch.tril(torch.ones(T, T))
weights = torch.zeros((T, T))

# 2. 遮蔽未来：将右上角变为负无穷
weights = weights.masked_fill(tril == 0, float('-inf'))  # 将矩阵中0元素的地方替换为负无穷

# 3. 通过 Softmax 归一化
# 此时每一行的和为 1，且非零部分的值完全相等（即 1/t）
weights = F.softmax(weights, dim=-1)

# 4. 执行融合
xbow3 = weights @ x

print("Softmax 产生的权重矩阵:\n", weights)
print("\n方案 3 (Softmax) 与方案 2 (直接归一化) 是否一致:")
print(torch.allclose(xbow, xbow3, atol=1e-7))

Softmax 产生的权重矩阵:
 tensor([[1.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000],
        [0.3333, 0.3333, 0.3333]])

方案 3 (Softmax) 与方案 2 (直接归一化) 是否一致:
True


**深度理解**：为什么初始化为 0 和 $-\infty$？

这两步是实现“因果掩码”（Causal Mask）的标准做法：

1.  **全 0 初始化**：代表在“注意力”分配之前，我们假设历史中的每一个词对当前时刻的影响力是**等同**的。
2.  **$-\infty$ 的妙用**：
    *   在数学上，$\text{Softmax}(x_i) = \frac{e^{x_i}}{\sum e^{x_j}}$。
    *   对于那些我们不想让模型看到的“未来时刻”，我们将 $x_i$ 设为 $-\infty$。
    *   由于 $e^{-\infty} = 0$，这些未来时刻在求和时贡献为 0，最终得到的概率也绝对为 0。

**总结**：这一步确保了模型在训练时不会“偷看”答案（即预测第 $t$ 个词时，不能看到第 $t+1$ 个词）。

### 3.6 **进阶**：从“简单平均”到“单头自注意力”

在 3.4 节中，我们通过矩阵乘法实现了**历史平均**。现在我们要升级它，让模型自己决定该“注意”哪些历史。

#### 1. 核心进化：Query, Key 和 Value
我们不再使用死板的平均权重，而是引入三个线性变换：
- **Query (Q)**: “我在寻找什么？”（当前时刻的意图）
- **Key (K)**: “我包含了什么信息？”（每个历史时刻的索引）
- **Value (V)**: “如果我被选中，我会贡献什么？”（实际传递的特征内容）

#### 2. 注意力计算流程
1.  **算权重 (Affinity)**：计算 $Q$ 和 $K$ 的点积。如果某个历史时刻的 Key 与当前的 Query 匹配度高，它就会获得更高的权重。
2.  **归一化**：使用 Softmax 让权重和为 1。
3.  **提内容**：用权重去对 **Value (V)** 进行加权求和，而不是对原始输入 $X$ 求平均。

#### 3. 最终公式
Transformer 中注意力机制的最终公式是：
$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

这就是为什么代码里最后一步是 `out = wei @ v`：它本质上是用“数据驱动的权重”提取出来的精华信息。

In [45]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 假设输入 x 的维度
B, T, C = 4, 8, 32
x = torch.randn(B, T, C)

# 1. 定义线性层
head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

# 2. 计算 Q, K, V
k = key(x)   # (B, T, head_size)
q = query(x) # (B, T, head_size)
v = value(x) # (B, T, head_size)

# 3. 计算注意力权重 (Affinity)
# TODO: 参照注意力公式计算权重
wei = q @ k.transpose(-2, -1)  # 交换张量K最后两个维度的位置：(batch_size, num_heads, seq_len_k, head_dim)

# TODO: 请对 wei 进行缩放 (Scaled Dot-Product Attention)
# 提示：将 wei 除以 head_size 的平方根。如果不缩放，Softmax 后的梯度会变得非常小。
wei = wei * (head_size ** -0.5)

# 4. 遮蔽未来信息与归一化
tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

# 5. 最后与 Value 融合
out = wei @ v

print("输出形状:", out.shape)
print("\n最后时刻的权重分布:\n", wei[-1, -1])

输出形状: torch.Size([4, 8, 16])

最后时刻的权重分布:
 tensor([0.1956, 0.1398, 0.0724, 0.1300, 0.1426, 0.1204, 0.1341, 0.0650],
       grad_fn=<SelectBackward0>)


观察一下：现在的 `wei` 矩阵不再是死板的 $1/t$ 了，而是由 $Q$ 和 $K$ 算出来的随机数（随后会被训练优化）。

**引导思考**：
如果两个 Token 及其相似（即它们的 $Q$ 和 $K$ 点积很大），它们之间的注意力权重会发生什么变化？这如何解决了 Bigram 模型“视野狭窄”的问题？

### 3.7 进阶：多头自注意力 (Multi-Head Attention)

为什么要多头？ 如果只有单头，模型一次只能学到一种关系（比如只关注“动词和名词”的关系）。但语言很复杂，我们需要模型同时关注多种关系（比如语法关系、长距离依赖、修饰关系等）。这就是多头自注意力的意义：让模型在不同的子空间里并行地学习。

我们可以把多头注意力想象成一堆并行运行的独立“头”。每个头都独立计算，最后我们将它们的输出拼接（Concatenate）在一起。

In [46]:
class Head(nn.Module):
    """ 一个单头自注意力层 """
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(C, head_size, bias=False)
        self.query = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))

    def forward(self, x):
        B, T, C = x.shape
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)

        # 计算注意力分数 (Affinity)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5) # (B, T, T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)

        # 融合 Value
        v = self.value(x)
        out = wei @ v # (B, T, head_size)
        return out

class MultiHeadAttention(nn.Module):
    """ 多个并行运行的自注意力头 """
    def __init__(self, num_heads, head_size):
        super().__init__()
        # TODO: 构建多头注意力模型
        # 提示: 使用 nn.ModuleList 存放多个独立的 Head
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        # 最后通过一个线性层投影回原始维度 (可选，但推荐)
        self.proj = nn.Linear(num_heads * head_size, C)

    def forward(self, x):
        # TODO: 拼接所有头的输出
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)

        return out

# 测试多头注意力
n_heads = 4
head_size = 8  # 4 * 8 = 32，正好回到了原始通道维度 C
mha = MultiHeadAttention(n_heads, head_size)
out = mha(x)

print(f"输入形状: {x.shape}")
print(f"多头输出形状: {out.shape}")

# 验证残差连接的可能性
print(f"它们能否相加? {'能' if x.shape == out.shape else '不能'}")

输入形状: torch.Size([4, 8, 32])
多头输出形状: torch.Size([4, 8, 32])
它们能否相加? 能


**引导思考**：
在 `MultiHeadAttention` 中，我们将 4 个 8 维的头拼接成了 32 维。这与直接使用 1 个 32 维的单头相比，参数量其实差不多。但为什么多头通常效果更好？

**引导思考**：
为什么我们在 `MultiHeadAttention` 的最后要加一个 `self.proj`？除了调整维度以适配残差连接（Residual Connection）之外，它在多头信息的融合上起到了什么作用？

### 3.9 深度观察：不同头之间的差异性

为了验证“多头是否真的在看不同的东西”，我们可以随机初始化一个多头层，观察两个不同头产生的权重矩阵 `wei`。你会发现，即便输入完全相同，它们的关注点也大相径庭。

In [47]:
# 观察不同头的注意力分布差异
head1 = mha.heads[0]
head2 = mha.heads[1]

with torch.no_grad():
    # 手动获取两个头的注意力矩阵
    k1, q1 = head1.key(x), head1.query(x)
    wei1 = (q1 @ k1.transpose(-2, -1)) * (head_size**-0.5)
    wei1 = F.softmax(wei1.masked_fill(tril[:T, :T] == 0, float('-inf')), dim=-1)

    k2, q2 = head2.key(x), head2.query(x)
    wei2 = (q2 @ k2.transpose(-2, -1)) * (head_size**-0.5)
    wei2 = F.softmax(wei2.masked_fill(tril[:T, :T] == 0, float('-inf')), dim=-1)

print("头 1 的最后时刻注意力分布:\n", wei1[0, -1])
print("\n头 2 的最后时刻注意力分布:\n", wei2[0, -1])
print("\n两个分布之间的差异 (欧氏距离):", torch.dist(wei1, wei2).item())

头 1 的最后时刻注意力分布:
 tensor([0.1380, 0.1038, 0.1207, 0.1317, 0.1212, 0.1019, 0.1388, 0.1441])

头 2 的最后时刻注意力分布:
 tensor([0.0822, 0.1138, 0.0660, 0.0828, 0.1665, 0.2576, 0.0882, 0.1428])

两个分布之间的差异 (欧氏距离): 1.0526785850524902


## Phase 4: Transformer 其余组件的实现
### 练习 4.1: 实现前馈网络 (Feed-Forward Network)

在 Transformer 块中，Self-Attention 负责聚合上下文信息，而 Feed-Forward 层负责对每个位置的向量进行独立地非线性变换。

In [ ]:
class FeedForward(nn.Module):
    """ 一个简单的线性层，后跟一个非线性激活函数 """
    def __init__(self, n_embed):
        super().__init__()
        self.net = nn.Sequential(
            # TODO: 实现第一层线性变换，将维度从 n_embed 扩大到 4 * n_embed
            # TODO: 添加 ReLU 激活函数
            # TODO: 实现第二层线性变换，将维度缩回到 n_embed
            # TODO: 添加一个投影层的 Dropout (概率设为 0.2)
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(0.2),
        )

    def forward(self, x):
        return self.net(x)

# 测试 FFN
ffn = FeedForward(C)
out_ffn = ffn(out)
print(f"FFN 输入形状: {out.shape}")
print(f"FFN 输出形状: {out_ffn.shape}")

**引导思考**: 为什么在 FFN 中，我们通常要把中间层的维度扩大（比如扩大到 4 倍），而不是直接保持 n_embed 不变？这种“先扩张再压缩”的结构在直觉上是为了做什么？

A: 这种“升维-激活-降维”的结构本质上是给模型提供了一个更大的特征空间。在 $4 \times n_{embed}$$4 \times n_{embed}$ 的高维空间中，模型能够学习到更加精细和非线性的特征组合，就像是在一个更大的草稿本上作画，然后再总结提炼回原始维度。这也被称为“逐位置前馈”（Point-wise Feed-Forward）。

### 练习 4.2: 层归一化 (Layer Normalization)

在 Transformer 中，我们在每个子层（Attention 和 FFN）之前都会应用 LayerNorm。这有助于稳定训练过程。

In [ ]:
class LayerNorm(nn.Module):
    """
    LayerNorm 的简化实现。注意它与 BatchNorm 的区别：
    BatchNorm 是在 Batch 维度归一化，而 LayerNorm 是在 Channel 维度归一化。
    """
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        # TODO: 用nn.Parameter初始化两个可学习参数：gamma (缩放) 和 beta (平移)
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))

    def forward(self, x):
        # TODO: 计算 x 在最后一个维度上的均值和方差
        # TODO: 执行归一化：(x - mean) / sqrt(var + eps)
        # TODO: 应用 gamma 和 beta 进行仿射变换
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True)
        eps = 1e-8
        x = (x - mean) / torch.sqrt(var + eps)
        x = x * self.gamma + self.beta

        return x

# 测试 LayerNorm
module = LayerNorm(C)
x_norm = module(x)
print(f"归一化后的均值 (应接近 0): {x_norm.mean().item():.4f}")
print(f"归一化后的标准差 (应接近 1): {x_norm.std().item():.4f}")

### 练习 4.3: 组装 Transformer Block

现在我们迎来了整个架构最激动人心的时刻：**整合 Transformer Block**。

一个标准的 Transformer Block 就像一块**乐高积木**，它把我们之前写的组件按以下顺序拼起来：
1.  **层归一化 (LayerNorm)**
2.  **多头自注意力 (Multi-Head Attention)**
3.  **残差连接 (+x)**
4.  **层归一化 (LayerNorm)**
5.  **前馈网络 (Feed-Forward)**
6.  **残差连接 (+x)**

**核心设计**：
- **Pre-Norm 结构**：在现代 Transformer (如 GPT-2) 中，我们通常在进入子层（Attention/FFN）之前进行归一化。这已被证明能让梯度流更稳定，使得训练深层网络变得容易。
- **残差连接 (Residual Connection)**：公式为 `x = x + sublayer(x)`。

**引导思考**：
为什么我们要使用残差连接？如果没有这个简单的加法，当我们堆叠 12 层甚至 96 层（如 GPT-3）时，模型在反向传播（Backpropagation）计算导数时会发生什么？

In [ ]:
class Block(nn.Module):
    """ Transformer Block: 通信 (Attention) + 计算 (FFN) """

    def __init__(self, n_embed, n_head):
        # n_embed: embedding 维度, n_head: 我们想要的头数
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)

    def forward(self, x):
        # TODO: 实现 Pre-Norm 结构下的残差连接
        # 1. x = x + Attention(LayerNorm(x))
        # 2. x = x + FFN(LayerNorm(x))
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

# 测试 Block
block = Block(n_embed=C, n_head=4)
out_block = block(x)
print(f"输入形状: {x.shape}")
print(f"Block 输出形状: {out_block.shape}")

## Phase 5: 最终集成 - 构建 GPT 语言模型

现在我们将之前的 `Block` 堆叠起来，并添加 **Positional Embedding**（位置嵌入），因为 Transformer 的注意力机制本身是“位置无关”的，我们需要告诉模型字符在序列中的具体位置。

In [ ]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size):
        super().__init__()
        # 1. Token 嵌入表
        self.token_embedding_table = nn.Embedding(vocab_size, n_embed)
        # 2. 位置 嵌入表 (让模型知道字符的顺序)
        self.position_embedding_table = nn.Embedding(block_size, n_embed)
        # 3. 堆叠多个 Transformer Block
        self.blocks = nn.Sequential(*[Block(n_embed, n_head) for _ in range(n_layer)])
        # 4. 最后的层归一化
        self.ln_f = LayerNorm(n_embed)
        # 5. 语言模型头 (从特征空间映射回词表大小)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # tok_emb: (B, T, C), pos_emb: (T, C)
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb # 融合内容信息和位置信息

        x = self.blocks(x)    # 通过所有 Block
        x = self.ln_f(x)      # 最终归一化
        logits = self.lm_head(x) # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            # 限制上下文长度不能超过 block_size
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] # 只关注最后一个时刻
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# 参数定义与初始化
n_layer = 4
gpt_model = GPTLanguageModel(vocab_size, C, n_heads, n_layer, block_size)
print(f"GPT 模型参数量: {sum(p.numel() for p in gpt_model.parameters())/1e3:.1f} K")

### 深度解析：位置嵌入（Position Embedding）的物理本质

在 Transformer 中，Attention 机制是完全“位置无关”的——就像把一袋子词扔进搅拌机。为了找回语序，我们引入了位置嵌入。以下是三个核心逻辑：

#### 1. 为什么是“相加”而不是“拼接”？
从直觉上看，拼接（Concatenate）似乎更能保留信息的独立性，但会显著增加维度。**相加（Addition）** 的巧妙之处在于利用了高维空间的**稀疏性**：
*   **数学原理**：在高维空间中，随机向量几乎总是正交的。模型可以学习将某些维度用于存储语义，而将另一些维度用于存储位置。由于线性变换的叠加性 $W(x + p) = Wx + Wp$，后续的线性层完全有能力解耦这两者。
*   **物理比喻**：语义是“颜色”，位置是“亮度”。我们将两者叠加，模型通过“滤镜”（权重矩阵）依然能分辨出颜色和亮度。

#### 2. 学习目标：从随机到“有序流形”
在初始化时，`position_embedding_table` 的每一行都是随机的，模型对“第 1 位”和“第 8 位”毫无感知。但在训练过程中：
*   模型需要通过 Attention 捕捉邻近词。为了让点积 $Q 	imes K$ 在近距离处更高，模型会迫使相邻位置的向量在向量空间中**靠得更近**，或者形成某种特定的周期性轨迹。
*   最终，这些向量会形成一个**低维流形**（类似于螺旋线），使得模型能感知“相对位移”。

#### 3. 局限性：绝对位置的“墙”
由于我们使用的是 `nn.Embedding(block_size, n_embed)`，这是一种**绝对位置编码**。模型只能认识索引在 $0$ 到 `block_size-1` 之间的位置。如果推理时输入第 9 个词，模型将无法处理，因为它从未见过对应的“位置身份证”。这就是为什么现代模型（如 Llama）转向旋转位置编码（RoPE）以获得更好的外推能力。

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 获取训练前的 Position Embedding 权重
pos_weights = gpt_model.position_embedding_table.weight.detach().cpu().numpy()

plt.figure(figsize=(10, 4))
sns.heatmap(pos_weights, cmap='viridis')
plt.title("Position Embedding Matrix (Initial State)")
plt.xlabel("Embedding Dimension (C)")
plt.ylabel("Position Index (T)")
plt.show()

**引导思考**：随着训练进行，你认为相邻位置（如 0 和 1）的向量会变得更接近还是更疏远？

## Phase 6: 训练与评估

我们将定义一个 `estimate_loss` 函数，它会暂时将模型设为评估模式 (`eval()`)，计算多个 batch 的平均 loss，然后再切回训练模式 (`train()`)。这是工业界的标准做法，可以过滤掉训练过程中的随机噪声。

In [ ]:
eval_iters = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'
gpt_model.to(device)

@torch.no_grad()
def estimate_loss():
    out = {}
    gpt_model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split, batch_size)
            X, Y = X.to(device), Y.to(device)
            logits, loss = gpt_model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    gpt_model.train()
    return out

# 重新定义优化器
optimizer = torch.optim.AdamW(gpt_model.parameters(), lr=1e-3, betas=(0.9, 0.9), eps=1e-7)

for iter in range(10000):
    # 1. 定期评估 Loss
    if iter % 1000 == 0 or iter == 9999:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # 2. 训练采样
    xb, yb = get_batch('train', batch_size)
    xb, yb = xb.to(device), yb.to(device)

    # 3. 梯度下降
    logits, loss = gpt_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

### 验收时刻：生成文本

模型训练完成后，我们可以让它从一个空字符（ID 0）开始，看看它是否学会了莎士比亚的遣词造句。

In [ ]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(gpt_model.generate(context, max_new_tokens=500)[0].tolist()))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 获取训练后的 Position Embedding 权重
pos_weights = gpt_model.position_embedding_table.weight.detach().cpu().numpy()

plt.figure(figsize=(10, 4))
sns.heatmap(pos_weights, cmap='viridis')
plt.title("Position Embedding Matrix (Trained State)")
plt.xlabel("Embedding Dimension (C)")
plt.ylabel("Position Index (T)")
plt.show()

### 进阶 1：探索 BPE Tokenization

更好的Token: 字符级（Character-level）虽然简单，但它是最“低效”的。目前大模型（如 GPT-4, Llama）的标准答案是 BPE (Byte Pair Encoding, 字节对编码)。

为什么字符级不够好？
- 序列太长：每个字母占一个位置，处理一本书需要百万级长度，而 Transformer 的计算量随长度平方级增长。
- 语义碎片化：'t', 'h', 'e' 本身没意义，只有组合成 'the' 才有意义。字符级强制模型去学习这些基础组合，浪费了参数量。

**什么是 BPE？**

BPE 是一种子词（Subword）算法。它从字符开始，不断将出现频率最高的相邻对合并。

例如：t 和 h 经常在一起，就把它们合并成一个新 Token th。
th 和 e 经常在一起，再合并成 the。
结果：常见的词（the, apple）变成单个 Token，不常见的词（unfamiliar）被切分成子词（un-familiar）。这完美平衡了词表大小和序列长度。

我们将对比字符级编码与 OpenAI GPT-4 使用的 `cl100k_base` 编码。观察同一个句子在不同编码下的 Token 数量。

In [ ]:
import tiktoken

# 准备测试文本
test_sentence = "To be, or not to be, that is the question."

# 1. 字符级编码 (我们之前的方案)
char_encoded = encode(test_sentence)

# 2. GPT-4 BPE 编码
enc = tiktoken.get_encoding("cl100k_base")
bpe_encoded = enc.encode(test_sentence)

print(f"原始文本长度: {len(test_sentence)}")
print(f"字符级 Token 数量: {len(char_encoded)}")
print(f"GPT-4 BPE Token 数量: {len(bpe_encoded)}")
print(f"\nBPE 编码内容: {bpe_encoded}")
print(f"BPE 还原结果: {[enc.decode([t]) for t in bpe_encoded]}")

### 为什么这很重要？

1. **压缩比**：你可以看到 BPE 的 Token 数量远少于字符数。这意味着在相同的 `block_size` 下，BPE 编码的模型能“读”到更长范围的上下文。
2. **效率**：模型每一层计算一次 Token 的表示。Token 越少，推理速度越快。
3. **跨语言**：BPE 可以处理任何语言，包括中文（通常一个中文字符对应一个或多个子词 Token）。

### 进阶 2: 扩展与超参数优化 (Scaling & Optimization)

现在我们要尝试训练一个稍大一点的模型。为了应对更深的网络，我们将：
1.  **增加模型规模**：增加层数和嵌入维度。
2.  **管理超参数**：将配置集中管理。
3.  **学习率预热与衰减 (Optional)**：虽然这里我们先保持简单，但在大型训练中这是标准操作。

In [ ]:
import tiktoken
import torch
import torch.nn as nn
from torch.nn import functional as F

# 1. 切换到 BPE 编码器并重新准备数据
enc = tiktoken.get_encoding("cl100k_base")
bpe_vocab_size = enc.n_vocab
data = torch.tensor(enc.encode(text), dtype=torch.long)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

# 2. 超参数配置
batch_size = 32
block_size = 128
max_iters = 5000
learning_rate = 3e-4
eval_interval = 500
n_embed = 256
n_head = 4
n_layer = 6
dropout = 0.2

# 3. 重新定义所有组件以确保参数匹配
class Head(nn.Module):
    def __init__(self, n_embed, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        wei = q @ k.transpose(-2, -1) * (k.shape[-1]**-0.5)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        return wei @ v

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size, n_embed):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embed, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embed)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.heads], dim=-1))

class Block(nn.Module):
    def __init__(self, n_embed, n_head):
        super().__init__()
        head_size = n_embed // n_head
        self.sa = MultiHeadAttention(n_head, head_size, n_embed)
        self.ffwd = FeedForward(n_embed)
        self.ln1 = LayerNorm(n_embed)
        self.ln2 = LayerNorm(n_embed)
    def forward(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ffwd(self.ln2(x))
        return x

In [ ]:
# 4. 初始化模型
final_model = GPTLanguageModel(bpe_vocab_size, n_embed, n_head, n_layer, block_size)
final_model.to(device)

print(f"Final BPE-GPT 参数量: {sum(p.numel() for p in final_model.parameters())/1e6:.2f} M")

# 5. 训练循环
optimizer = torch.optim.AdamW(final_model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0:
        final_model.eval()
        eval_loss = 0
        with torch.no_grad():
            for _ in range(10):
                ix = torch.randint(0, len(val_data) - block_size, (batch_size,))
                xb = torch.stack([val_data[i:i+block_size] for i in ix]).to(device)
                yb = torch.stack([val_data[i+1:i+block_size+1] for i in ix]).to(device)
                _, loss = final_model(xb, yb)
                eval_loss += loss.item()
        print(f"Step {iter}: Val Loss {eval_loss/10:.4f}")
        final_model.train()

    ix = torch.randint(0, len(train_data) - block_size, (batch_size,))
    xb = torch.stack([train_data[i:i+block_size] for i in ix]).to(device)
    yb = torch.stack([train_data[i+1:i+block_size+1] for i in ix]).to(device)

    logits, loss = final_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

    if iter % 100 == 0:
        print(f"Iter {iter}: Train Loss {loss.item():.4f}")

阶段性成果验收 (Inference Check)

让我们看看在训练了若干步后，使用 BPE 的 27M 模型生成的文本。注意，由于词表巨大且参数量较多，前期的训练可能会更多地集中在学习单词拼写上。

In [ ]:
# 切换到评估模式
final_model.eval()

# 从空字符（或者是莎士比亚剧本中常见的开头）开始生成
# 在 BPE 中，我们直接用 enc.encode("\n") 作为起始符
context = torch.tensor([enc.encode("\n")], dtype=torch.long, device=device)

# 生成 200 个 Token (这大约相当于之前的 600-800 个字符)
generated_tokens = final_model.generate(context, max_new_tokens=200)[0].tolist()

print("--- 27M BPE-GPT 阶段性生成结果 ---")
print(enc.decode(generated_tokens))
print("----------------------------------")

# 切回训练模式继续后续训练（如果需要）
# final_model.train()

## Phase 7: 进阶训练策略 
### 7.1 学习率调度器 (Learning Rate Scheduler)

在训练大规模模型时，固定学习率往往会导致两个问题：
1. **初期不稳**：刚开始梯度很大，直接用高学习率可能导致 Loss 飞掉。
2. **后期震荡**：训练末期如果步子还是那么大，模型很难钻进那个最深的“谷底”。

我们将实现一个典型的 **Warmup + Cosine Decay** 调度器：
*   **Warmup**：前几百步将学习率从 0 线性增加到设定值。
*   **Cosine Decay**：之后按照余弦曲线将学习率降至一个很小的最小值。

In [ ]:
import math

# 学习率调度配置
warmup_iters = 200
lr_decay_iters = max_iters # 通常等于总迭代次数
min_lr = 3e-5 # 最终衰减到的最小值，通常是 learning_rate / 10

def get_lr(it):
    # 1) 线性预热阶段 (linear warmup)
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    # 2) 超过衰减阶段，返回最小学习率
    if it > lr_decay_iters:
        return min_lr
    # 3) 余弦衰减阶段
    decay_ratio = (it - warmup_iters) / (lr_decay_iters - warmup_iters)
    assert 0 <= decay_ratio <= 1
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio)) # 从 1 降到 0
    return min_lr + coeff * (learning_rate - min_lr)

# 演示学习率曲线
lrs = [get_lr(i) for i in range(max_iters)]
plt.plot(lrs)
plt.title("Learning Rate Schedule (Warmup + Cosine Decay)")
plt.xlabel("Iterations")
plt.ylabel("Learning Rate")
plt.show()

### 7.2 权重衰减 (Weight Decay) 与 梯度裁剪 (Gradient Clipping)

为了训练更加稳健的 27M 模型，我们引入以下策略：
- **Decoupled Weight Decay**: 仅对维度 $\ge 2$ 的权重矩阵应用 decay。
- **Grad Clipping**: 限制梯度的总模长，防止梯度爆炸。

In [ ]:
def get_optimizer(model, weight_decay, learning_rate, betas):
    # 提取所有需要梯度的参数
    param_dict = {pn: p for pn, p in model.named_parameters() if p.requires_grad}
    
    # 区分需要 decay 和不需要 decay 的组
    # 规则：2D 权重矩阵（Linear.weight）需要 decay，1D（bias, LayerNorm.weight/beta）不需要
    decay_params = [p for n, p in param_dict.items() if p.dim() >= 2]
    nodecay_params = [p for n, p in param_dict.items() if p.dim() < 2]
    
    optim_groups = [
        {'params': decay_params, 'weight_decay': weight_decay},
        {'params': nodecay_params, 'weight_decay': 0.0}
    ]
    
    optimizer = torch.optim.AdamW(optim_groups, lr=learning_rate, betas=betas)
    return optimizer

# 配置参数
weight_decay = 0.1
grad_clip = 1.0

# 重新初始化优化器
optimizer = get_optimizer(final_model, weight_decay, learning_rate, (0.9, 0.95))
print(f"优化器已更新：启用了权重衰减 ({weight_decay})")

### 7.3 整合：完全体训练循环 (Complete Training Loop)

现在我们将所有的技巧：**BPE Tokenization**、**27M Transformer**、**LR Scheduler**、**Weight Decay** 和 **Grad Clipping** 整合在一起。你可以运行这个循环来观察模型的收敛。

In [ ]:
# 1. 重新初始化模型
final_model = GPTLanguageModel(bpe_vocab_size, n_embed, n_head, n_layer, block_size)
final_model.to(device)

# 2. 重新配置优化器（包含权重衰减）
weight_decay = 0.1
grad_clip = 1.0
optimizer = get_optimizer(final_model, weight_decay, learning_rate, (0.9, 0.9))

print(f"开始完全体训练（参数量: {sum(p.numel() for p in final_model.parameters())/1e6:.2f} M）...")

for iter in range(max_iters):
    # 1. 动态调整学习率 (Warmup + Cosine Decay)
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # 2. 获取数据与前向传播
    ix = torch.randint(0, len(train_data) - block_size, (batch_size,))
    xb = torch.stack([train_data[i:i+block_size] for i in ix]).to(device)
    yb = torch.stack([train_data[i+1:i+block_size+1] for i in ix]).to(device)

    logits, loss = final_model(xb, yb)

    # 3. 反向传播与优化
    optimizer.zero_grad(set_to_none=True)
    loss.backward()

    # 4. 梯度裁剪：防止梯度爆炸，维持训练稳定性
    torch.nn.utils.clip_grad_norm_(final_model.parameters(), grad_clip)

    optimizer.step()

    if iter % 200 == 0 or iter == max_iters - 1:
        print(f"Iter {iter}: Loss {loss.item():.4f}, Current LR: {lr:.2e}")

print("训练完成！")

### 7.4 跨模型评价指标：Perplexity & BPC

由于 BPE 和字符级模型的词表大小不同，直接比较 Loss 是不公平的。我们引入两个通用指标：
1. **Perplexity (PPL)**: $\exp(\text{loss})$。它反映了模型在选择下一个 Token 时，平均在多少个选项中犹豫。
2. **Bits Per Character (BPC)**: $\frac{\text{Total Loss} \times \log_2(e)}{\text{Total Characters}}$。它将 Loss 归一化到原始字符维度。**这是衡量“Tokenizer + 模型预测能力”综合优劣的标准**。无论词表如何变化，BPC 越低，说明模型对底层信息的压缩和预测越高效。

In [ ]:
import math

@torch.no_grad()
def evaluate_metrics(model, data_source, num_batches=50):
    model.eval()
    total_loss = 0
    total_tokens = 0
    
    for _ in range(num_batches):
        ix = torch.randint(0, len(data_source) - block_size, (batch_size,))
        xb = torch.stack([data_source[i:i+block_size] for i in ix]).to(device)
        yb = torch.stack([data_source[i+1:i+block_size+1] for i in ix]).to(device)
        
        _, loss = model(xb, yb)
        total_loss += loss.item() * (batch_size * block_size)
        total_tokens += (batch_size * block_size)

    avg_loss = total_loss / total_tokens
    perplexity = math.exp(avg_loss)
    
    # 计算 BPC (假设每个 Token 对应的平均字符数可以通过数据集比例估算)
    # 莎士比亚原始字符数 / BPE Token 数
    chars_per_token = 1115394 / len(data)
    bpc = avg_loss * math.log2(math.e) / chars_per_token
    
    model.train()
    return {"Loss": avg_loss, "Perplexity": perplexity, "BPC": bpc}

# 执行评估
metrics = evaluate_metrics(final_model, val_data)
print(f"验证集评估结果:")
print(f"- Cross Entropy Loss: {metrics['Loss']:.4f}")
print(f"- Perplexity: {metrics['Perplexity']:.2f}")
print(f"- BPC (Bits Per Character): {metrics['BPC']:.4f}")

结果数据解读：
- Perplexity ($\approx 1700$)：这代表模型在预测下一个 BPE Token 时，平均在 1710 个可能的单词/子词之间犹豫。对于 10 万大小的词表，这已经是一个巨大的进步了。

- 物理意义：BPC 2.9 意味着模型认为，平均只需要约 2.9 个“比特”的信息就能确定莎士比亚剧本中的一个字符。作为对比，如果我们完全随机猜（1/65 的概率），BPC 是 $\log_2(65) \approx 6.0 \log_2(65) \approx 6.0$。你的模型已经成功从数据中挖掘出了一半以上的信息规律。

- 横向对比：在莎士比亚数据集上，顶级的字符级 GPT 通常能达到 1.4~1.6 BPC。当前的 2.9 说明模型已经入门，但由于训练步数还不够（刚跑完初始阶段），它还没能抓到更深层的文学修辞和逻辑。

---

## Phase 8: 下游任务迁移能力 (Downstream Task Transferability)

大型语言模型（LLMs）的真正强大之处在于它们的**泛化能力**。一个在大量文本数据上预训练的模型，可以在不进行大量修改的情况下，通过微调（Fine-tuning）或上下文学习（In-context Learning）的方式，很好地适应各种下游任务，例如：

1.  **文本分类**：判断文本情感、新闻分类等。
2.  **问答系统**：从文本中提取答案。
3.  **文本摘要**：将长篇文档总结成简短摘要。
4.  **机器翻译**：将一种语言翻译成另一种语言。
5.  **代码生成/补全**：辅助程序员编写代码。

**为什么预训练的模型具有这种能力？**

通过预测下一个 Token，模型被迫学习了语言的深层结构、语义关系、事实知识和世界模型。这些学习到的“表征”（Representations）是通用的，可以作为各种特定任务的良好起点。

**如何实现迁移？**

*   **微调 (Fine-tuning)**：在预训练模型之上添加一个简单的任务特定层（例如一个线性分类器），然后在特定任务的小数据集上继续训练模型。
*   **上下文学习 (In-context Learning)**：在不修改模型参数的情况下，通过在输入中提供任务描述和几个示例，引导模型完成新任务。这是大型 LLMs（如 GPT-3）展现出的强大能力。

**思考**: 字符级或 BPE 级别的语言模型，在预训练阶段是如何“学会”语言的语法和语义规则的？

### 8.1: 文本分类 (Text Classification) 示例

我们将演示如何将预训练的 `GPTLanguageModel` 适配到**文本分类**任务。假设我们想进行情感分类，将文本归类为正面或负面。核心思想是：

1.  **重用预训练的特征提取器**：保留 `GPTLanguageModel` 的所有层（除了最后的 `lm_head`），利用它们生成上下文相关的文本嵌入。
2.  **添加分类头**：在模型顶部添加一个简单的线性层，将文本嵌入映射到分类类别（例如，2 个类别表示正面/负面）。
3.  **冻结或微调**：可以选择冻结（freeze）预训练的层，只训练分类头，这在小数据集上很有效；或者解冻部分或所有层进行微调。

我们将构建一个 `GPTClassifier` 类，它继承 `GPTLanguageModel` 的大部分结构，并替换其 `lm_head`。

In [ ]:
class GPTClassifier(GPTLanguageModel):
    def __init__(self, vocab_size, n_embed, n_head, n_layer, block_size, num_classes=2):
        super().__init__(vocab_size, n_embed, n_head, n_layer, block_size)
        # 移除原有的语言模型头
        del self.lm_head

        # 添加一个新的分类头
        self.classification_head = nn.Linear(n_embed, num_classes)

    def forward(self, idx, targets=None):
        B, T = idx.shape

        # 沿用 GPTLanguageModel 的前向传播直到 ln_f
        tok_emb = self.token_embedding_table(idx)
        pos_emb = self.position_embedding_table(torch.arange(T, device=idx.device))
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)

        # 对于分类任务，我们通常只取序列中最后一个 token 的表示
        # 或者使用一个特殊的 [CLS] token 的表示 (这里我们简化为最后一个 token)
        # 假设最后一个 token 的表示聚合了整个序列的信息
        last_token_embedding = x[:, -1, :]

        # 通过分类头获得分类 logits
        logits = self.classification_head(last_token_embedding) # (B, num_classes)

        loss = None
        if targets is not None:
            # 计算分类损失 (例如交叉熵损失)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

# 1. 实例化分类模型
# 我们将使用与之前 GPT 模型相同的配置，但添加 2 个输出类别
num_classes = 2 # 例如：正面/负面情感
classifier_model = GPTClassifier(bpe_vocab_size, n_embed, n_head, n_layer, block_size, num_classes)
classifier_model.to(device)

# 2. 从预训练的 GPT 模型加载权重
# 假设我们已经训练好了 final_model，将其权重加载到 classifier_model 中
# 注意：lm_head 的权重不会被加载，因为它已经被替换。
# 同时，我们需要处理 lm_head 在 state_dict 中的键，通常通过过滤或重命名来实现。
pretrained_state_dict = final_model.state_dict()

# 过滤掉 lm_head 相关的键
filtered_state_dict = {k: v for k, v in pretrained_state_dict.items() if 'lm_head' not in k}

# 加载权重，strict=False 允许 state_dict 中有未匹配的键 (即 classification_head)
classifier_model.load_state_dict(filtered_state_dict, strict=False)
print("预训练模型权重已加载到分类器模型。")

# 3. 冻结大部分层，只训练分类头
for name, param in classifier_model.named_parameters():
    if 'classification_head' not in name: # 除了分类头，其他层都冻结
        param.requires_grad = False
    else:
        param.requires_grad = True

print("冻结了除分类头以外的所有层。")
print("需要训练的参数:")
for name, param in classifier_model.named_parameters():
    if param.requires_grad:
        print(f"- {name}")

# 4. 模拟一个分类任务的前向传播
# 假设有一个 Batch 的输入文本 (B, T) 和对应的标签 (B,)
# 这里我们创建一个模拟数据
dummy_input_idx = torch.randint(0, bpe_vocab_size, (batch_size, block_size)).to(device) # 随机 token IDs
dummy_labels = torch.randint(0, num_classes, (batch_size,)).to(device) # 随机标签 (0或1)

logits, loss = classifier_model(dummy_input_idx, dummy_labels)
print(f"\n模拟分类 logits 形状: {logits.shape}") # (Batch, num_classes)
print(f"模拟分类 Loss: {loss.item():.4f}")

# 现在你可以用一个新的优化器和分类数据来训练 classifier_model 了。
# 例如:
# classification_optimizer = torch.optim.AdamW(classifier_model.parameters(), lr=1e-4)
# for epoch in range(num_epochs):
#     # 获取分类任务的 batch (text_batch, label_batch)
#     logits, loss = classifier_model(text_batch, label_batch)
#     loss.backward()
#     classification_optimizer.step()
#     classification_optimizer.zero_grad()

#### 快速开始分类任务微调
运行下方代码块，通过模拟数据对分类头进行 5 步快速微调，并对测试文本进行预测：

In [ ]:
# 1. 准备实际样本数据
sample_texts = [
    "I love this Shakespeare play, it is beautiful!", # Positive
    "The tragedy was moving and well written.",     # Positive
    "This performance was terrible and boring.",    # Negative
    "I did not enjoy the show at all.",             # Negative
    "A masterpiece of English literature."          # Positive
]
# 1代表正面，0代表负面
sample_labels = torch.tensor([1, 1, 0, 0, 1], device=device)

# 2. 简单微调循环
optimizer = torch.optim.AdamW(classifier_model.classification_head.parameters(), lr=1e-3)
classifier_model.train()

print("正在使用实际文本样本微调分类头...")
for epoch in range(20):
    # 将文本转换为 Token IDs 并填充/截断到 block_size
    input_ids = []
    for text in sample_texts:
        tokens = enc.encode(text)
        # 简单的填充或截断逻辑
        if len(tokens) > block_size: tokens = tokens[:block_size]
        else: tokens += [0] * (block_size - len(tokens))
        input_ids.append(tokens)
    
    xb = torch.tensor(input_ids, device=device)
    yb = sample_labels

    logits, loss = classifier_model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
    if (epoch + 1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# 3. 实时预测示例
classifier_model.eval()
with torch.no_grad():
    test_text = "This play is a complete waste of time."
    test_tokens = enc.encode(test_text)
    # 确保长度匹配
    if len(test_tokens) > block_size: test_tokens = test_tokens[:block_size]
    else: test_tokens += [0] * (block_size - len(test_tokens))
    
    test_idx = torch.tensor([test_tokens], device=device)
    logits, _ = classifier_model(test_idx)
    probs = torch.softmax(logits, dim=-1)
    prediction = torch.argmax(probs).item()

print(f"\n测试文本: {test_text}")
print(f"预测结果: {'正面' if prediction == 1 else '负面'} (置信度: {probs[0, prediction]:.2%})")

### 8.2: 构建对话型模型 (Conversational Model)

大型语言模型最直观的应用之一就是作为**对话型代理**（Conversational Agent）。这基本上是其**文本生成能力**的直接延伸。核心思想是：

1.  **维持上下文**：模型需要记住整个对话历史，才能生成相关的回复。
2.  **增量生成**：每次用户输入后，将用户输入加入到历史中，然后让模型继续生成新的内容作为回复。
3.  **停止条件**：需要定义模型何时停止生成（例如，生成了句号、换行符，或者达到最大长度）。

我们可以将直接使用之前训练好的 `final_model` 来进行对话。

**注意**: 即使是像 `final_model` 这样的小型模型，其生成的对话可能并不完全连贯或有逻辑，但它会努力模仿语言的结构和风格。

In [ ]:
# 切换到评估模式
final_model.eval()

# BPE 编码器
enc = tiktoken.get_encoding("cl100k_base")

def chat(model, max_new_tokens=100, stop_token='\n'):
    conversation_history = ""
    print("--- 开始对话 (输入 'exit' 结束) ---")

    while True:
        user_input = input("你: ")
        if user_input.lower() == 'exit':
            break

        # 将用户输入添加到历史中
        conversation_history += user_input + "\n"

        # 编码对话历史
        context_tokens = enc.encode(conversation_history)
        # 限制上下文长度不超过 block_size
        context_tokens = context_tokens[-block_size:]
        context = torch.tensor([context_tokens], dtype=torch.long, device=device)

        # 生成回复
        generated_tokens_tensor = model.generate(context, max_new_tokens=max_new_tokens)
        generated_tokens = generated_tokens_tensor[0].tolist()

        # 解码所有生成的 Token，然后提取新增的部分
        full_generated_text = enc.decode(generated_tokens)
        # 找出新生成的文本，从历史长度开始
        model_response = full_generated_text[len(conversation_history):]

        # 寻找停止 Token 并截断
        if stop_token in model_response:
            model_response = model_response.split(stop_token, 1)[0] + stop_token
        
        # 打印模型回复
        print(f"模型: {model_response.strip()}")
        
        # 更新对话历史，包含模型回复
        conversation_history += model_response

    print("--- 对话结束 ---")

# 运行对话函数
# 可以调整 max_new_tokens 来控制模型回复的长度
chat(final_model, max_new_tokens=80, stop_token='\n')

#### 8.2.1: 对话模型微调示例

我们将模拟一个小型对话数据集，并展示如何使用它对 `final_model` 进行微调。数据格式通常是 `user_turn` + `model_response`，这样模型就能学会预测下一个回复。

In [ ]:
import random

# 模拟一个小型对话数据集
# 每个元组包含 (对话历史, 预期模型回复)
# 为了简化，这里每个条目都假设是独立的对话片段，
# 实际中，需要构建完整的对话链
dialogue_dataset = [
    ("Hello, how are you?", "I'm doing well, thank you! How can I assist you today?"),
    ("Tell me about Shakespeare.", "William Shakespeare was an English playwright, poet, and actor, widely regarded as the greatest writer in the English language."),
    ("What is the meaning of life?", "That is a profound philosophical question that has been debated for centuries. There is no single universally accepted answer."),
    ("Can you write a poem?", "Certainly! What topic would you like the poem to be about?"),
    ("I need help with my code.", "I can help with coding. What kind of problem are you facing?"),
    ("Who is your creator?", "I am a large language model, trained by Google."),
    ("How old are you?", "I do not have an age in the human sense, as I am an AI."),
    ("Goodbye.", "Goodbye! Have a great day."),
    ("What's your favorite book?", "As an AI, I don't read books or have preferences like humans do, but I can process information from countless texts!"),
    ("Thank you!", "You're most welcome! Is there anything else I can help with?"),
    ("Where are you from?", "I exist as a computer program, so I don't have a physical origin in the way humans do."),
    ("What do you think of this play?", "I can analyze the text of the play if you provide it. What aspects are you interested in?"),
    ("I am feeling sad.", "I'm sorry to hear that. Is there anything specific you'd like to talk about or any information I can provide?"),
    ("Happy to see you.", "It's good to 'see' you too! How may I assist you?")
]

# 将对话数据转换为 token ID 序列
def prepare_dialogue_batch(batch_size):
    xb_dialogue, yb_dialogue = [], []
    for _ in range(batch_size):
        # 随机选择一个对话片段
        user_turn, model_response = random.choice(dialogue_dataset)
        # 拼接用户输入和模型响应，用换行符分隔作为训练目标
        full_text = user_turn + "\n" + model_response + "\n"
        
        # 编码整个序列
        encoded_full_text = enc.encode(full_text)
        
        # 确保序列长度不超过 block_size
        if len(encoded_full_text) < block_size + 1:
            # x 是除了最后一个 token 的所有 token
            # y 是除了第一个 token 的所有 token (即 x 的下一个 token)
            x = encoded_full_text[:-1]
            y = encoded_full_text[1:]
            
            # 填充到 block_size
            padding_len = block_size - len(x)
            x = x + [0] * padding_len # 使用 0 (通常是 PAD token) 进行填充
            y = y + [0] * padding_len
            
            xb_dialogue.append(x)
            yb_dialogue.append(y)
        else:
            # 如果太长，就截断
            x = encoded_full_text[:block_size]
            y = encoded_full_text[1:block_size+1]
            xb_dialogue.append(x)
            yb_dialogue.append(y)

    return torch.tensor(xb_dialogue, dtype=torch.long, device=device), torch.tensor(yb_dialogue, dtype=torch.long, device=device)

# 测试数据准备
sample_xb, sample_yb = prepare_dialogue_batch(2)
print("Sample XB shape:", sample_xb.shape)
print("Sample YB shape:", sample_yb.shape)
print("Sample XB (decoded):", [enc.decode(x.tolist()) for x in sample_xb])
print("Sample YB (decoded):", [enc.decode(y.tolist()) for y in sample_yb])


#### 微调训练循环

我们将使用与之前类似的训练循环，但现在使用我们的模拟对话数据集。为了避免覆盖之前在莎士比亚数据上学到的知识，并且更专注于对话模式，我们可能会使用一个较小的学习率，或者只训练部分层（例如只解冻 `lm_head` 附近的一些层，或者不冻结任何层进行全面微调，取决于数据量）。

在这里，我们将不冻结任何层，让模型在对话数据上进行端到端微调。我们使用一个独立的优化器和训练迭代次数来区分这个微调过程。

In [ ]:
# 重新初始化一个优化器，用于对话微调
# 通常使用较小的学习率进行微调
conversation_learning_rate = 1e-5
conversation_optimizer = torch.optim.AdamW(final_model.parameters(), lr=conversation_learning_rate, betas=(0.9, 0.95))

# 设置微调迭代次数
micro_finetune_iters = 1000
grad_clip = 1.0

print("\n--- 开始对话模型微调 ---")
final_model.train()

for iter_ft in range(micro_finetune_iters):
    # 获取对话 Batch 数据
    xb_ft, yb_ft = prepare_dialogue_batch(batch_size)

    # 前向传播计算 Loss
    logits_ft, loss_ft = final_model(xb_ft, yb_ft)

    # 反向传播更新参数
    conversation_optimizer.zero_grad(set_to_none=True)
    loss_ft.backward()    
    torch.nn.utils.clip_grad_norm_(final_model.parameters(), grad_clip) # 梯度裁剪
    conversation_optimizer.step()

    if (iter_ft + 1) % 100 == 0:
        print(f"Finetune Iter {iter_ft+1}: Loss {loss_ft.item():.4f}")

print("对话模型微调完成！")


#### 使用微调后的模型进行对话

现在，我们再次运行 `chat` 函数，看看微调后的 `final_model` 在生成对话回复方面是否有改进。它应该会更倾向于生成类似模拟数据集中的回答模式。

In [ ]:
# 切换到评估模式
final_model.eval()

# 运行对话函数
# 可以调整 max_new_tokens 来控制模型回复的长度
chat(final_model, max_new_tokens=80, stop_token='\n')

# 切回训练模式（如果需要继续训练）
# final_model.train()